# Introducción al concepto de isocronas

En geografía, física o análisis espacial, muchas veces queremos representar zonas o líneas donde una cierta variable tiene el mismo valor.
Por ejemplo:
- En un mapa topográfico, las curvas de nivel conectan puntos con igual altura.
- En meteorología, las isobaras conectan puntos con igual presión atmosférica.
- En climatología, las isotermas muestran puntos con igual temperatura.

A este tipo de representación se la llama “isolinea” (del griego isos, igual, y linea, línea).
Cada tipo de isolínea representa una magnitud diferente, pero todas comparten la misma idea: conectar puntos equivalentes respecto a una medida o propiedad.

Dentro de ese mismo grupo de isolíneas, encontramos las isocronas, que unen puntos que tienen igual tiempo respecto a un punto de origen o evento.

Imaginemos que queremos saber hasta dónde podemos llegar desde un punto en un tiempo determinado: por ejemplo, todos los lugares a los que puedo llegar en 10 minutos en auto desde mi casa. La línea que conecta todos esos puntos se llama isocrona.

Por lo tanto, una isocrona representa una línea de igual tiempo de viaje. Del mismo modo que una curva de nivel en un mapa une puntos con igual altura, una isocrona une puntos con igual tiempo de desplazamiento.


### Ejemplo visual

Podemos imaginar una isocrona como una “curva de tiempo”:

- Si estamos en una ciudad y trazamos una línea de 5 minutos, otra de 10 y otra de 15 desde una estación de tren, obtenemos zonas concéntricas que muestran hasta dónde se puede llegar en esos intervalos. Esas fronteras temporales serían nuestras isocronas.
- Cada una muestra una “capa” de accesibilidad, de manera similar a cómo las curvas de nivel muestran diferentes alturas.

<img src="https://fiverr-res.cloudinary.com/images/q_auto,f_auto/gigs/355867848/original/2cf2ece57d4a6422e59808b6ff8b6473c28d7a97/do-isochrone-map-time-access-by-walking-cycling-or-driving.png" width="500">

### Aplicaciones

Aunque las isocronas se usan mucho en transporte y movilidad, el concepto puede aplicarse también a:
- Propagación de fenómenos naturales, como el avance de un incendio o una inundación.
- Difusión de información o señales, en redes o sistemas físicos.
- Modelos de expansión urbana o contagio, donde el “tiempo” representa velocidad de propagación.

En urbanismo o transporte, estas zonas permiten analizar:
- Accesibilidad (por ejemplo, qué barrios están a menos de 15 min de un hospital)
- Cobertura de servicios (escuelas, bomberos, transporte público)
- Análisis logístico (tiempos de entrega o respuesta)

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
from shapely.ops import unary_union
import networkx as nx
import matplotlib.pyplot as plt

# Desde el punto de vista computacional

Para calcular una isocrona, necesitamos un modelo de red:
- Los nodos representan ubicaciones (intersecciones, paradas, puntos del mapa).
- Las aristas representan caminos, con un peso (distancia o tiempo).
- Luego usamos algoritmos de caminos más cortos, como Dijkstra, para determinar qué nodos son alcanzables dentro de un umbral de tiempo o distancia.

El conjunto de todos esos nodos forma nuestra isocrona digital.

## Ejemplo:

In [ ]:
# Creamos un pequeño grafo de ejemplo

# Creamos un grafo dirigido
G = nx.DiGraph()

# Agregamos los nodos y aristas con "tiempo" como peso
edges = [
    ("A", "B", 5),
    ("A", "C", 7),
    ("B", "D", 6),
    ("C", "D", 4),
    ("C", "E", 5),
    ("D", "F", 8),
    ("E", "F", 6)
]

# Agregamos los edges con su peso
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

#
pos = {"A": (0,0), "B": (1,2), "C": (3,1),
       "D": (4,3), "E": (5,0), "F": (6,2)}

# Dibujamos el grafo
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=800, arrows=True)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'weight'))
plt.title("Red de ejemplo con tiempos de viaje (minutos)")
plt.show()

In [ ]:
# Distancia desde A a todos los nodos
nx.single_source_dijkstra_path_length(G, "A", weight='weight')

In [ ]:
# Distancia desde C a todos los nodos
nx.single_source_dijkstra_path_length(G, "C", weight='weight')

In [ ]:
# Distancia desde F a todos los nodos
nx.single_source_dijkstra_path_length(G, "F", weight='weight')

Supongamos que queremos saber a qué nodos puedo llegar desde A en 10 minutos o menos.

In [ ]:
# Calcular isocrona - retorna los nodos a una distancia límite
def compute_isochrone(graph, source, time_limit):
    # Calculamos la distancia mínima a cada nodo usando Dijkstra
    lengths = nx.single_source_dijkstra_path_length(graph, source, weight='weight')

    # Filtramos los nodos dentro del tiempo límite
    reachable = {node: dist for node, dist in lengths.items() if dist <= time_limit}
    return reachable

# Calcular isocrona desde el punto A, con un límite de 10
isochrone = compute_isochrone(G, source="A", time_limit=10)
print(isochrone)

In [ ]:
print("Nodos alcanzables desde A en 10 minutos o menos:")
for node, dist in isochrone.items():
    print(f"{node}: {dist} minutos")

In [ ]:
# Visualizar la isocrona
# Los nodos en verde son los que pertenecen a la isocrona de 4 minutos.

color_map = ["lightgreen" if node in isochrone else "lightgray" for node in G.nodes()]

nx.draw(G, pos, with_labels=True, node_color=color_map, node_size=800, arrows=True)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'weight'))
plt.title("Isochrona desde A (≤4 min)")
plt.show()

In [ ]:
# Nodos alcanzables en distintos tiempos
for t in [5, 10, 15]:
    reachable = compute_isochrone(G, "A", t)
    print(f"Isocrona {t} min: {list(reachable.keys())}")

Qué aristas están involucradas en la isocrona?     
A veces no queremos solo los nodos, sino también qué aristas están “activas” dentro de la isocrona, es decir: aquellas cuya longitud total acumulada desde el origen no supera el límite.

In [ ]:
def get_isochrone_edges(graph, source, time_limit):
    lengths = nx.single_source_dijkstra_path_length(graph, source, weight='weight')
    edges_in_iso = []
    for u, v, data in graph.edges(data=True):
        # Si ambos nodos son alcanzables y su distancia acumulada no supera el límite
        if u in lengths and v in lengths:
            if lengths[u] <= time_limit and lengths[v] <= time_limit:
                edges_in_iso.append((u, v, data['weight']))
    return edges_in_iso

iso_edges = get_isochrone_edges(G, "A", 10)
print("Aristas dentro de la isocrona (≤10 min):")
for u, v, w in iso_edges:
    print(f"{u} → {v} ({w} min)")

In [ ]:
# Dibujamos todo el grafo
nx.draw(G, pos, with_labels=True, node_color="lightgray", node_size=800, arrows=True)

# Resaltamos las aristas dentro de la isocrona
nx.draw_networkx_edges(G, pos, edgelist=[(u, v) for u, v, _ in iso_edges],
                       edge_color="green", width=3)

plt.title("Aristas dentro de la isocrona (≤6 min desde A)")
plt.show()

Generamos una subred con los nodos y ejes dentro de la isocrona.

In [ ]:
isochrone

In [ ]:
# Filtramos los nodos dentro del tiempo límite
nodos_alcanzados = set(isochrone.keys())
nodos_alcanzados

In [ ]:
# Creamos un subgrafo con esos nodos
subG = G.subgraph(nodos_alcanzados).copy()
print(subG.nodes())

# Esto crea un subgrafo (subG) que contiene solo los nodos y aristas totalmente dentro del límite temporal.

In [ ]:
subG

In [ ]:
# Dibujamos la subred resultante
nx.draw(G, pos, with_labels=True, node_color="lightgray", node_size=800, arrows=True)
nx.draw(subG, pos, with_labels=True, node_color="lightgreen", node_size=800, arrows=True, width=2)
plt.title("Subred dentro de la isocrona (≤10 min desde A)")
plt.show()

# Isocronas desde dos nodos

In [ ]:
# Posiciones para plot
pos = {"A": (0,0), "B": (1,2), "C": (3,1),
       "D": (4,3), "E": (5,0), "F": (6,2)}

# Dibujamos el grafo
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=800, arrows=True)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'weight'))
plt.title("Red de ejemplo con tiempos de viaje (minutos)")
plt.show()

## Calculo de la isocrona un nodo a la vez

In [ ]:
# Calcular isocronas

# Parámetros
origenes = ["A","C"]
limite_tiempo = 10

# Calcular distancias (Dijkstra) para cada origen
isocronas = {}
for origen in origenes:
    distancias = nx.single_source_dijkstra_path_length(G, origen, weight="weight")
    nodos_alcanzables = {n: t for n, t in distancias.items() if t <= limite_tiempo}
    isocronas[origen] = nodos_alcanzables

# Mostrar resultados
for origen, nodos_tiempos in isocronas.items():
    print(f"\nIsocrona desde nodo {origen} (≤ {limite_tiempo} minutos):")
    for nodo, tiempo in nodos_tiempos.items():
        print(f"  Nodo {nodo}: tiempo {tiempo}")

## Calculo de la isocrona a partir de todos a la vez

In [ ]:
# Multi-source Dijkstra
distancias, caminos = nx.multi_source_dijkstra(G, sources=origenes, weight="weight")

# Mostrar distancias mínimas desde cualquiera de los nodos origen
for nodo, d in distancias.items():
    print(f"Nodo {nodo}: distancia mínima desde {origenes} = {d}")

In [ ]:
# Nodos alcanzables
nodos_alcanzables = [n for n,d in distancias.items() if d <= limite_tiempo]

# Aristas alcanzables
aristas_alcanzables = []
for u,v,d in G.edges(data=True):
    peso = d['weight']
    # Verificamos si existe algún origen desde el cual se puede alcanzar u y luego v
    for origen in origenes:
        dist_u = nx.single_source_dijkstra_path_length(G, origen, weight="weight").get(u,float('inf'))
        dist_v = nx.single_source_dijkstra_path_length(G, origen, weight="weight").get(v,float('inf'))
        # Solo incluimos la arista si podemos llegar desde el origen y recorrer la arista dentro del límite
        if dist_u != float('inf') and dist_v != float('inf'):
          if dist_u + peso <= limite_tiempo or dist_v + peso <= limite_tiempo:
              aristas_alcanzables.append((u,v))
              break
aristas_alcanzables

In [ ]:
# Plot
plt.figure(figsize=(8,6))

# Grafo completo de fondo
nx.draw(G, pos, node_color='lightgray', edge_color='lightgray', with_labels=True, node_size=800)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G,'weight'))

# Nodos alcanzables
nx.draw_networkx_nodes(G, pos, nodelist=nodos_alcanzables, node_color='orange', node_size=800)

# Aristas alcanzables
nx.draw_networkx_edges(G, pos, edgelist=aristas_alcanzables, edge_color='red', width=3)

plt.title(f"Isochrona combinada desde {origenes} con límite {limite_tiempo}")
plt.show()

# **Isocrona espacial**

In [ ]:
# Crear el grafo con coordenadas simuladas (ejemplo didáctico)

G = nx.Graph()

# Nodos con coordenadas simuladas (x, y) tipo UTM o long/lat
nodes = {
    "A": (0, 0),
    "B": (1, 2),
    "C": (3, 1),
    "D": (4, 3),
    "E": (5, 0),
    "F": (6, 2)
}

# Agregar nodos
for name, (x, y) in nodes.items():
    G.add_node(name, x=x, y=y)

# Agregar aristas con pesos (ej: tiempo en minutos)
edges = [
    ("A", "B", 5),
    ("A", "C", 7),
    ("B", "D", 6),
    ("C", "D", 4),
    ("C", "E", 5),
    ("D", "F", 8),
    ("E", "F", 6)
]
G.add_weighted_edges_from(edges)

In [ ]:
# Calcular la isocrona desde un nodo origen

origen = "A"
limite_tiempo = 12  # minutos

# Calcular la distancia mínima (peso acumulado)
distancias = nx.single_source_dijkstra_path_length(G, origen, weight="weight")

# Filtrar nodos dentro del límite
nodos_isocrona = [n for n, d in distancias.items() if d <= limite_tiempo]

# Subgrafo (solo la parte accesible dentro del límite)
subG = G.subgraph(nodos_isocrona).copy()

In [ ]:
# Visualizar la red original y la isocrona

plt.figure(figsize=(8,6))

# Posiciones reales (coordenadas geográficas simuladas)
pos = {n: (G.nodes[n]['x'], G.nodes[n]['y']) for n in G.nodes()}

# Dibujar todo el grafo en gris
nx.draw(G, pos, node_color='lightgray', edge_color='lightgray', with_labels=True, node_size=800)

# Dibujar la subred de la isocrona
nx.draw(subG, pos, node_color='orange', edge_color='red', with_labels=True, node_size=800, width=2)

# Agregar etiquetas de pesos a las aristas (de toda la red o solo de la subred)
edge_labels = nx.get_edge_attributes(G, 'weight')  # o subG si querés solo las de la isocrona
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='blue', font_size=10)

plt.title(f"Isochrona desde nodo {origen} (≤ {limite_tiempo} minutos)")
plt.show()

In [ ]:
subG.nodes['A']

In [ ]:
# Crear GeoDataFrames (nodos y aristas)

# Nodos
data_nodos = []
for n in subG.nodes():
    x, y = subG.nodes[n]['x'], subG.nodes[n]['y']
    d = distancias[n]
    data_nodos.append({"nodo": n, "weight_acumulado": d, "geometry": Point(x, y)})

gdf_nodos = gpd.GeoDataFrame(data_nodos, crs="EPSG:4326")

gdf_nodos.head()

In [ ]:
# Aristas
data_aristas = []
for u, v, data in subG.edges(data=True):
    x1, y1 = subG.nodes[u]['x'], subG.nodes[u]['y']
    x2, y2 = subG.nodes[v]['x'], subG.nodes[v]['y']
    weight = data['weight']
    # Tomamos el máximo del peso acumulado entre ambos extremos
    weight_acum = max(distancias[u], distancias[v])
    data_aristas.append({
        "nodo_origen": u,
        "nodo_destino": v,
        "weight": weight,
        "weight_acumulado": weight_acum,
        "geometry": LineString([(x1, y1), (x2, y2)])
    })

gdf_aristas = gpd.GeoDataFrame(data_aristas, crs="EPSG:4326")

gdf_aristas.head()

In [ ]:
# Crear el polígono del área isócrona
# Crear un polígono convexo a partir de los puntos alcanzados
polygon = gdf_nodos.unary_union.convex_hull
gdf_poligono = gpd.GeoDataFrame([{"geometry": polygon, "origen": origen, "tiempo_max": limite_tiempo}], crs="EPSG:4326")
gdf_poligono.head()

In [ ]:
# Visualizar en mapa

fig, ax = plt.subplots(figsize=(8,6))
gdf_poligono.plot(ax=ax, color="lightblue", alpha=0.4, edgecolor="blue")
gdf_aristas.plot(ax=ax, color="red", linewidth=2)
gdf_nodos.plot(ax=ax, color="orange", markersize=80)

for idx, row in gdf_nodos.iterrows():
    ax.text(row.geometry.x + 0.05, row.geometry.y + 0.05, f"{row['nodo']} ({row['weight_acumulado']})")

plt.title("Área de la isocrona y subred alcanzable")
plt.xlabel("X (longitud simulada)")
plt.ylabel("Y (latitud simulada)")
plt.show()

In [ ]:
# Crear el polígono del área isócrona (con buffer)

# Crear un buffer alrededor de cada eje alcanzado
# El valor del buffer depende de la escala de tus coordenadas (aquí ejemplo genérico)
buffer_dist = 0.5  # distancia del buffer, ajustar según unidades del grafo

# Generar un buffer para cada eje
gdf_aristas_buffer = gdf_aristas.copy()
gdf_aristas_buffer["buffer"] = gdf_aristas_buffer.geometry.buffer(buffer_dist)

# Unir todos los buffers para formar un área continua
union_buffer = gdf_aristas_buffer["buffer"].unary_union

# Crear GeoDataFrame con el área resultante
gdf_poligono = gpd.GeoDataFrame(
    [{"geometry": union_buffer, "origen": origen, "tiempo_max": limite_tiempo}],
    crs="EPSG:4326"
)
gdf_poligono.head()

In [ ]:
# Visualizar en mapa

fig, ax = plt.subplots(figsize=(8,6))
gdf_poligono.plot(ax=ax, color="lightblue", alpha=0.4, edgecolor="blue")
gdf_aristas.plot(ax=ax, color="red", linewidth=2)
gdf_nodos.plot(ax=ax, color="orange", markersize=80)

for idx, row in gdf_nodos.iterrows():
    ax.text(row.geometry.x + 0.05, row.geometry.y + 0.05, f"{row['nodo']} ({row['weight_acumulado']})")

plt.title("Área de la isocrona y subred alcanzable")
plt.xlabel("X (longitud simulada)")
plt.ylabel("Y (latitud simulada)")
plt.show()

In [ ]:
# Crear el polígono del área isócrona (con buffer)

# Crear un buffer alrededor de cada eje alcanzado
# El valor del buffer depende de la escala de tus coordenadas (aquí ejemplo genérico)
buffer_dist = 2  # distancia del buffer, ajustar según unidades del grafo
buffer_dist_back = -1.8

# Generar un buffer para cada eje
gdf_aristas_buffer = gdf_aristas.copy()
gdf_aristas_buffer["geometry"] = gdf_aristas_buffer.geometry.buffer(buffer_dist)

# Unir todos los buffers para formar un área continua
gdf_aristas_buffer["geometry"] = gdf_aristas_buffer["geometry"].unary_union
gdf_aristas_buffer["geometry"]  = gdf_aristas_buffer.geometry.buffer(buffer_dist_back)
union_buffer = gdf_aristas_buffer["geometry"].unary_union

# Crear GeoDataFrame con el área resultante
gdf_poligono = gpd.GeoDataFrame(
    [{"geometry": union_buffer, "origen": origen, "tiempo_max": limite_tiempo}],
    crs="EPSG:4326"
)

In [ ]:
# Visualizar en mapa

fig, ax = plt.subplots(figsize=(8,6))
gdf_poligono.plot(ax=ax, color="lightblue", alpha=0.4, edgecolor="blue")
gdf_aristas.plot(ax=ax, color="red", linewidth=2)
gdf_nodos.plot(ax=ax, color="orange", markersize=80)

for idx, row in gdf_nodos.iterrows():
    ax.text(row.geometry.x + 0.05, row.geometry.y + 0.05, f"{row['nodo']} ({row['weight_acumulado']})")

plt.title("Área de la isocrona y subred alcanzable")
plt.xlabel("X (longitud simulada)")
plt.ylabel("Y (latitud simulada)")
plt.show()

# Calcular un “semi-eje” según el tiempo restante

In [ ]:
origen = "A"
limite_tiempo = 12

# Calcular tiempo acumulado para cada nodo
distancias = nx.single_source_dijkstra_path_length(G, origen, weight="weight")

# Calcular tiempo para cada arista
edge_data = []
for u, v, d in G.edges(data=True):
    t_origen = distancias.get(u, float('inf'))
    peso = d['weight']
    edge_data.append({
        "origen": u,
        "destino": v,
        "peso": peso,
        "tiempo_inicio": t_origen,
        "tiempo_fin": t_origen + peso
    })

# Separar aristas completas y semi-ejes
complete_edges = []
semi_edges = []

for e in edge_data:
    if e["tiempo_fin"] <= limite_tiempo:
        complete_edges.append(e)
    elif e["tiempo_inicio"] < limite_tiempo:
        porcentaje = (limite_tiempo - e["tiempo_inicio"]) / e["peso"]
        semi_edges.append({**e, "porcentaje": porcentaje})

# Crear geometrías
# Nodos alcanzables
gdf_nodos = gpd.GeoDataFrame([
    {"nodo": n, "weight_acumulado": distancias[n], "geometry": Point(G.nodes[n]['x'], G.nodes[n]['y'])}
    for n in distancias if distancias[n] <= limite_tiempo
], crs="EPSG:4326")

# Aristas completas
gdf_aristas = gpd.GeoDataFrame([
    {"origen": e["origen"], "destino": e["destino"], "weight": e["peso"],
     "weight_acumulado": e["tiempo_fin"],
     "geometry": LineString([Point(G.nodes[e["origen"]]['x'], G.nodes[e["origen"]]['y']),
                             Point(G.nodes[e["destino"]]['x'], G.nodes[e["destino"]]['y'])])}
    for e in complete_edges
], crs="EPSG:4326")

# Semi-ejes
gdf_semi = gpd.GeoDataFrame([
    {"origen": e["origen"], "destino": e["destino"], "weight": e["peso"],
     "weight_acumulado": e["tiempo_inicio"] + (e["porcentaje"]*e["peso"]),
     "geometry": LineString([
         Point(G.nodes[e["origen"]]['x'], G.nodes[e["origen"]]['y']),
         Point(
             G.nodes[e["origen"]]['x'] + (G.nodes[e["destino"]]['x'] - G.nodes[e["origen"]]['x'])*e["porcentaje"],
             G.nodes[e["origen"]]['y'] + (G.nodes[e["destino"]]['y'] - G.nodes[e["origen"]]['y'])*e["porcentaje"]
         )
     ])}
    for e in semi_edges
], crs="EPSG:4326")

# Crear polígono final con buffer
buffer_dist = 0.3
buffers_nodos = gdf_nodos.geometry.buffer(buffer_dist)
buffers_edges = list(gdf_aristas.geometry.buffer(buffer_dist/2)) + list(gdf_semi.geometry.buffer(buffer_dist/2))
union_geom = unary_union(list(buffers_nodos) + buffers_edges)
gdf_poligono = gpd.GeoDataFrame([{"geometry": union_geom}], crs="EPSG:4326")

# Visualización
fig, ax = plt.subplots(figsize=(8,6))
gdf_poligono.plot(ax=ax, color="lightblue", alpha=0.4, edgecolor="blue")
gdf_aristas.plot(ax=ax, color="red", linewidth=2)
gdf_semi.plot(ax=ax, color="orange", linewidth=2)
gdf_nodos.plot(ax=ax, color="orange", markersize=80)

for idx, row in gdf_nodos.iterrows():
    ax.text(row.geometry.x + 0.05, row.geometry.y + 0.05, f"{row['nodo']} ({row['weight_acumulado']:.1f})")

plt.title("Isochrona con semi-ejes (grafo completo A–F)")
plt.show()